In [ ]:
from pathlib import Path
import sys

src_dir = next(p for p in [Path.cwd(), *Path.cwd().parents]
               if (p / "common.py").is_file() and (p / "feature_preference").is_dir())
sys.path.insert(0, str(src_dir))


In [ ]:
%load_ext autoreload
%autoreload 2
import os
import numpy as np
import random
import pandas as pd
from PIL import Image
from tqdm.auto import tqdm
from dataset_utils import combine_and_mask
import matplotlib.pyplot as plt
import cv2
import common

In [ ]:
import torchvision

places_dir = '/home/dataset/Places365'

torchvision.datasets.Places365(download=False, split='train-standard', small=True, root=places_dir)
torchvision.datasets.Places365(download=False, split='val', small=True, root=places_dir)

In [ ]:
bird_ratio = [0.03, 0.05, 0.07, 0.09]
# bird_ratio = [0.75]
cub_dir = '/home/dataset/CUB_200_2011'
output_dir = '/home/dataset/waterbird_notest_nspurious'
# output_dir = '/home/dataset/waterbird_multiple_color_background'
# output_dir = '/home/dataset/waterbird_simple_color_background'
# output_dir = '/home/dataset/waterbird_single_background'
target_places = [
    ['bamboo_forest', 'forest/broadleaf'],  # Land backgrounds
    ['ocean', 'lake/natural']]              # Water backgrounds

val_frac = 0             # What fraction of the training data to use as validation
# background_predictability = 0.70
background_predictability = 0.50
min_mask_ratio = max(bird_ratio)       # min mask ratio

In [ ]:
bias = 0.8
def get_max_bounding_box(mask):
    # Find indices where mask is 1
    y_indices, x_indices = np.where(mask >= bias)
    # If no 1s are found, return None
    if len(y_indices) == 0:
        return (0, 0, 0, 0)

    # Get min and max coordinates
    min_y, max_y = np.min(y_indices), np.max(y_indices)
    min_x, max_x = np.min(x_indices), np.max(x_indices)

    return (min_y, min_x, max_y, max_x)

def get_ratio(mask):
    bbox = get_max_bounding_box(mask)
    min_y, min_x, max_y, max_x = bbox
    height = max_y - min_y + 1
    width = max_x - min_x + 1
    area = height * width
    mask_area = np.sum(mask >= bias)
    ratio = mask_area / area

    return ratio

In [ ]:
images_path = os.path.join(cub_dir, 'images.txt')

df = pd.read_csv(
    images_path,
    sep=" ",
    header=None,
    names=['img_id', 'img_filename'],
    index_col='img_id')

mask_ratios = []
filtered_img_ids = []

for img_id, row in tqdm(df.iterrows()):
    img_filename = row['img_filename']
    seg_path = os.path.join(cub_dir, 'segmentations', img_filename.replace('.jpg','.png'))
    try:
        # seg_np = np.asarray(cv2.cvtColor(cv2.imread(seg_path), cv2.COLOR_BGR2RGB)) / 255
        seg_np = np.asarray(cv2.imread(seg_path, cv2.IMREAD_GRAYSCALE)) / 255
        ratio = get_ratio(seg_np)
        mask_ratios.append(ratio)


        if ratio < min_mask_ratio:
            filtered_img_ids.append(img_id)
            print(f"Filtering {img_filename} with mask ratio {ratio:.2f}")
    except Exception as e:
        print(f"Error processing {img_filename}: {e}")
        filtered_img_ids.append(img_id)

print(f"remove under mask ratio {min_mask_ratio}: {len(filtered_img_ids)}")
df = df.drop(filtered_img_ids)
print(f"after filtering: {len(df)}")

### Set up labels of waterbirds vs. landbirds
# We consider water birds = seabirds and waterfowl.
species = np.unique([img_filename.split('/')[0].split('.')[1].lower() for img_filename in df['img_filename']])
water_birds_list = [
    'Albatross', # Seabirds
    'Auklet',
    'Cormorant',
    'Frigatebird',
    'Fulmar',
    'Gull',
    'Jaeger',
    'Kittiwake',
    'Pelican',
    'Puffin',
    'Tern',
    'Gadwall', # Waterfowl
    'Grebe',
    'Mallard',
    'Merganser',
    'Guillemot',
    'Pacific_Loon'
]

In [ ]:
plt.plot(np.sort(mask_ratios))
plt.axhline(min_mask_ratio, color='r', linestyle='--')

In [ ]:
common.set_seed(0)
water_birds = {}
for species_name in species:
    water_birds[species_name] = 0
    for water_bird in water_birds_list:
        if water_bird.lower() in species_name:
            water_birds[species_name] = 1
species_list = [img_filename.split('/')[0].split('.')[1].lower() for img_filename in df['img_filename']]
df['y'] = [water_birds[species] for species in species_list]

### Assign train/tesst/valid splits
# In the original CUB dataset split, split = 0 is test and split = 1 is train
# We want to change it to
# split = 0 is train,
# split = 1 is val,
# split = 2 is test

train_test_df = pd.read_csv(
    os.path.join(cub_dir, 'train_test_split.txt'),
    sep=" ",
    header=None,
    names=['img_id', 'split'],
    index_col='img_id')

df = df.join(train_test_df, on='img_id')
test_ids = df.loc[df['split'] == 0].index
train_ids = np.array(df.loc[df['split'] == 1].index)
val_ids = np.random.choice(
    train_ids,
    size=int(np.round(val_frac * len(train_ids))),
    replace=False)

if 'notest' in output_dir:
    df.loc[train_ids, 'split'] = 0
    df.loc[val_ids, 'split'] = 1
    df.loc[test_ids, 'split'] = 0
else:
    df.loc[train_ids, 'split'] = 0
    df.loc[val_ids, 'split'] = 1
    df.loc[test_ids, 'split'] = 2

### Assign confounders (place categories)

# Confounders are set up as the following:
# Y = 0, C = 0: background_predictability
# Y = 0, C = 1: 1 - background_predictability
# Y = 1, C = 0: 1 - background_predictability
# Y = 1, C = 1: background_predictability

df['place'] = 0
train_ids = np.array(df.loc[df['split'] == 0].index)
val_ids = np.array(df.loc[df['split'] == 1].index)
test_ids = np.array(df.loc[df['split'] == 2].index)
for split_idx, ids in enumerate([train_ids, val_ids, test_ids]):
    for y in (0, 1):
        if y == 0:
            pos_fraction = 1 - background_predictability
        else:
            pos_fraction = background_predictability
        subset_df = df.loc[ids, :]
        y_ids = np.array((subset_df.loc[subset_df['y'] == y]).index)
        pos_place_ids = np.random.choice(
            y_ids,
            size=int(np.round(pos_fraction * len(y_ids))),
            replace=False)
        df.loc[pos_place_ids, 'place'] = 1

for split, split_label in [(0, 'train'), (1, 'val'), (2, 'test')]:
    print(f"{split_label}:")
    split_df = df.loc[df['split'] == split, :]
    print(f"waterbirds are {np.mean(split_df['y']):.3f} of the examples")
    print(f"y = 0, c = 0: {np.mean(split_df.loc[split_df['y'] == 0, 'place'] == 0):.3f}, n = {np.sum((split_df['y'] == 0) & (split_df['place'] == 0))}")
    print(f"y = 0, c = 1: {np.mean(split_df.loc[split_df['y'] == 0, 'place'] == 1):.3f}, n = {np.sum((split_df['y'] == 0) & (split_df['place'] == 1))}")
    print(f"y = 1, c = 0: {np.mean(split_df.loc[split_df['y'] == 1, 'place'] == 0):.3f}, n = {np.sum((split_df['y'] == 1) & (split_df['place'] == 0))}")
    print(f"y = 1, c = 1: {np.mean(split_df.loc[split_df['y'] == 1, 'place'] == 1):.3f}, n = {np.sum((split_df['y'] == 1) & (split_df['place'] == 1))}")

### Assign places to train, val, and test set
place_ids_df = pd.read_csv(
    os.path.join(places_dir, 'categories_places365.txt'),
    sep=" ",
    header=None,
    names=['place_name', 'place_id'],
    index_col='place_id')

target_place_ids = []

for idx, target_places in enumerate(target_places):
    place_filenames = []

    for target_place in target_places:
        target_place_full = f'/{target_place[0]}/{target_place}'
        assert (np.sum(place_ids_df['place_name'] == target_place_full) == 1)
        target_place_ids.append(place_ids_df.index[place_ids_df['place_name'] == target_place_full][0])
        print(f'train category {idx} {target_place_full} has id {target_place_ids[idx]}')

        # Read place filenames associated with target_place
        place_filenames += [
            f'/{target_place[0]}/{target_place}/{filename}' for filename in os.listdir(
                os.path.join(places_dir, 'data_256', target_place[0], target_place))
            if filename.endswith('.jpg')]

    random.shuffle(place_filenames)

    # Assign each filename to an image
    indices = (df.loc[:, 'place'] == idx)
    assert len(place_filenames) >= np.sum(indices),\
        f"Not enough places ({len(place_filenames)}) to fit the dataset ({np.sum(df.loc[:, 'place'] == idx)})"
    df.loc[indices, 'place_filename'] = place_filenames[:np.sum(indices)]

In [ ]:
place_path = os.path.join(places_dir, 'data_256', 'f/forest/broadleaf/00004816.jpg')
place_forest = np.asarray(cv2.cvtColor(cv2.imread(place_path), cv2.COLOR_BGR2RGB))
plt.imshow(place_forest)
plt.show()
print(0)

place_path = os.path.join(places_dir, 'data_256', 'l/lake/natural/00004253.jpg')
place_ocean = np.asarray(cv2.cvtColor(cv2.imread(place_path), cv2.COLOR_BGR2RGB))
plt.imshow(place_ocean)
plt.show()
print(1)

In [ ]:
for pref_ratio in bird_ratio:
    dataset_name = f'{pref_ratio}'
    output_subfolder = os.path.join(output_dir, dataset_name)
    os.makedirs(output_subfolder, exist_ok=True)

    df.to_csv(os.path.join(output_subfolder, 'metadata.csv'))

    ratio = []
    real_ratios = []
    for i in tqdm(df.index):
        # Load bird image and segmentation
        img_path = os.path.join(cub_dir, 'images', df.loc[i, 'img_filename'])
        seg_path = os.path.join(cub_dir, 'segmentations', df.loc[i, 'img_filename'].replace('.jpg','.png'))
        img_np = np.asarray(cv2.cvtColor(cv2.imread(img_path), cv2.COLOR_BGR2RGB))
        seg_np = np.asarray(cv2.imread(seg_path, cv2.IMREAD_GRAYSCALE)) / 255 >= bias
        min_y, min_x, max_y, max_x = get_max_bounding_box(seg_np)
        original_ratio = get_ratio(seg_np)
        bird = img_np[min_y:max_y+1, min_x:max_x+1]
        bird_mask = seg_np[min_y:max_y+1, min_x:max_x+1]
        # Load place background
        # Skip front /

        if "waterbird_single_background" in output_dir:
            if df.loc[i, 'place'] == 0:
                place = place_forest
            else:
                place = place_ocean
        elif "waterbird_simple_color_background" in output_dir:
            if df.loc[i, 'place'] == 0:
                place = np.full((256, 256, 3), [0, 255, 0], dtype=np.uint8)
            else:
                place = np.full((256, 256, 3), [0, 0, 255], dtype=np.uint8)
        else:
            place_path = os.path.join(places_dir, 'data_256', df.loc[i, 'place_filename'][1:])
            place = np.asarray(cv2.cvtColor(cv2.imread(place_path), cv2.COLOR_BGR2RGB))

        combined_img, real_ratio = combine_and_mask(place, bird, bird_mask, original_ratio, pref_ratio)
        real_ratios.append(real_ratio)

        output_path = os.path.join(output_subfolder, df.loc[i, 'img_filename'])
        os.makedirs('/'.join(output_path.split('/')[:-1]), exist_ok=True)

        Image.fromarray(combined_img).save(output_path)
        if real_ratio < 0.005:
            print(output_path)

    plt.plot(np.sort(real_ratios))
    plt.axhline(pref_ratio, color='r', linestyle='--')
    plt.show()